# **KLASIFIKASI BERITA MENGGUNAKAN LATENT DIRICHLET ALLOCATION (LDA)**

## Load hasil preprocessing

In [1]:
import pandas as pd

df_tempo_processed_csv = pd.read_csv('/content/tempo_preprocessed.csv')
display(df_tempo_processed_csv)

,id_berita,judul_berita,isi_berita,kategori_berita,text,clean_text,tokens
0,2076486,Ketika Para Jenderal Ikut Defile di HUT ke-80 TNI,TENTARA Nasional Indonesia atau TNI menggelar ...,politik,Ketika Para Jenderal Ikut Defile di HUT ke-80 ...,ketika para jenderal ikut defile di hut ke tni...,"['jenderal', 'defile', 'hut', 'tni', 'tentara'..."
1,2076480,Prabowo Minta Semua Pesantren Didata setelah P...,PRESIDENPrabowoSubianto memerintahkan semua po...,politik,Prabowo Minta Semua Pesantren Didata setelah P...,prabowo minta semua pesantren didata setelah p...,"['prabowo', 'pesantren', 'didata', 'ponpes', '..."
2,2076479,Prabowo: Utamakan Kompetensi Prajurit Dibandin...,PRESIDEN Prabowo Subianto memerintahkan Pangli...,politik,Prabowo: Utamakan Kompetensi Prajurit Dibandin...,prabowo utamakan kompetensi prajurit dibanding...,"['prabowo', 'utamakan', 'kompetensi', 'prajuri..."
3,2076473,Kemenkomdigi: Permintaan Data ke TikTok Hanya ...,KEMENTERIAN Komunikasi dan Digital (Kemenkomdi...,politik,Kemenkomdigi: Permintaan Data ke TikTok Hanya ...,kemenkomdigi permintaan data ke tiktok hanya u...,"['kemenkomdigi', 'permintaan', 'data', 'tiktok..."
4,2076468,Megawati dan Jokowi Tak Hadir di HUT ke-80 TNI...,MANTAN Presiden Megawati Soekarnoputri dan Jok...,politik,Megawati dan Jokowi Tak Hadir di HUT ke-80 TNI...,megawati dan jokowi tak hadir di hut ke tni di...,"['megawati', 'jokowi', 'hadir', 'hut', 'tni', ..."
...,...,...,...,...,...,...,...
895,2073886,Ketika Harry Kane Memecahkan Rekor Gol Cristia...,ADA dua kondisi yang kini melekat padaHarry Ka...,sepakbola,Ketika Harry Kane Memecahkan Rekor Gol Cristia...,ketika harry kane memecahkan rekor gol cristia...,"['harry', 'kane', 'memecahkan', 'rekor', 'gol'..."
896,2073880,Peluang Timnas Indonesia Lewati Hadangan Arab ...,PENGAMAT sepak bola Tanah Air Kesit Budi Hando...,sepakbola,Peluang Timnas Indonesia Lewati Hadangan Arab ...,peluang timnas indonesia lewati hadangan arab ...,"['peluang', 'timnas', 'indonesia', 'lewati', '..."
897,2073852,Seperti Apa Ketajaman Cristiano Ronaldo Bersam...,"DALAM usia 40 tahun,Cristiano Ronaldomasih mam...",sepakbola,Seperti Apa Ketajaman Cristiano Ronaldo Bersam...,seperti apa ketajaman cristiano ronaldo bersam...,"['ketajaman', 'cristiano', 'ronaldo', 'nassr',..."
898,2073819,FIFA Jatuhkan Sanksi untuk Malaysia dan 7 Pema...,"BADAN sepak bola dunia,FIFA, menjatuhkan sanks...",sepakbola,FIFA Jatuhkan Sanksi untuk Malaysia dan 7 Pema...,fifa jatuhkan sanksi untuk malaysia dan pemain...,"['fifa', 'jatuhkan', 'sanksi', 'malaysia', 'pe..."


## Siapkan teks untuk LDA

In [2]:
df_tempo_processed_csv['tokens_list'] = df_tempo_processed_csv['tokens'].apply(eval)
df_tempo_processed_csv['tokens_str'] = df_tempo_processed_csv['tokens_list'].apply(lambda x: " ".join(x))

## Buat representasi “bag of words”

In [3]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer()
X_bow = vectorizer.fit_transform(df_tempo_processed_csv['tokens_str'])

## Jalankan LDA untuk menemukan topik

In [4]:
from sklearn.decomposition import LatentDirichletAllocation

lda_model = LatentDirichletAllocation(n_components=10, random_state=42)
X_topics = lda_model.fit_transform(X_bow)


## Lihat Hasik Topik

In [5]:
terms = vectorizer.get_feature_names_out()
num_top_words = 10

for idx, topic in enumerate(lda_model.components_):
    print(f"\nTopik {idx+1}:")
    print(", ".join([terms[i] for i in topic.argsort()[:-num_top_words - 1:-1]]))


Topik 1:
tiktok, data, oktober, digital, komdigi, pemerintah, akun, sistem, kementerian, pilihan

Topik 2:
pertandingan, gol, pemain, liga, laga, babak, kemenangan, tim, menit, bermain

Topik 3:
indonesia, pemain, games, sea, timnas, jakarta, indra, keluarga, oktober, sampah

Topik 4:
penerbangan, wisata, harga, iran, dunia, gram, kredit, emas, tiket, veto

Topik 5:
israel, gaza, trump, palestina, negara, anggota, hukum, dpr, kpk, korupsi

Topik 6:
gempa, gunung, kali, september, rumah, bmkg, aktivitas, kilometer, janice, pesisir

Topik 7:
indonesia, persen, oktober, triliun, september, saham, pasar, keuangan, laut, pemerintah

Topik 8:
kapal, jakarta, oktober, orang, hujan, korban, wilayah, pilihan, global, sumud

Topik 9:
tni, oktober, prabowo, hut, jakarta, jalan, indonesia, presiden, tanah, acara

Topik 10:
marquez, motogp, balapan, pembalap, posisi, marc, poin, dunia, indonesia, juara


In [6]:
df_topics = pd.DataFrame(X_topics, columns=[f"topik_{i+1}" for i in range(lda_model.n_components)])
df_final = pd.concat([df_tempo_processed_csv, df_topics], axis=1)

df_final[["judul_berita", "kategori_berita"] + [f"topik_{i+1}" for i in range(lda_model.n_components)]].head()

,judul_berita,kategori_berita,topik_1,topik_2,topik_3,topik_4,topik_5,topik_6,topik_7,topik_8,topik_9,topik_10
0,Ketika Para Jenderal Ikut Defile di HUT ke-80 TNI,politik,0.000470,0.000470,0.000470,0.000470,0.000470,0.000470,0.000470,0.014201,0.982043,0.000470
1,Prabowo Minta Semua Pesantren Didata setelah P...,politik,0.030815,0.000649,0.000649,0.000649,0.000650,0.000649,0.072570,0.794568,0.098150,0.000649
2,Prabowo: Utamakan Kompetensi Prajurit Dibandin...,politik,0.000426,0.000426,0.000426,0.000426,0.000426,0.000426,0.000426,0.000426,0.827622,0.168974
3,Kemenkomdigi: Permintaan Data ke TikTok Hanya ...,politik,0.684194,0.000495,0.000495,0.000495,0.000495,0.000495,0.311845,0.000495,0.000495,0.000495
4,Megawati dan Jokowi Tak Hadir di HUT ke-80 TNI...,politik,0.000532,0.000532,0.000532,0.000532,0.000532,0.000532,0.000532,0.000532,0.995212,0.000532


In [7]:
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report

X = X_topics
y = df_tempo_processed_csv["kategori_berita"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

nb = MultinomialNB()
nb.fit(X_train, y_train)
y_pred_nb = nb.predict(X_test)

print("=== Hasil Naive Bayes ===")
print(classification_report(y_test, y_pred_nb))

=== Hasil Naive Bayes ===
               precision    recall  f1-score   support

      ekonomi       0.41      0.32      0.36        22
      hiburan       0.43      0.14      0.21        22
        hukum       0.22      0.38      0.28        13
internasional       0.67      0.26      0.38        23
   lingkungan       0.41      0.64      0.50        22
     olahraga       0.36      0.28      0.31        18
     otomotif       0.67      0.88      0.76        16
      politik       0.28      0.32      0.30        25
    sepakbola       0.54      0.74      0.62        19

     accuracy                           0.42       180
    macro avg       0.44      0.44      0.41       180
 weighted avg       0.44      0.42      0.40       180



In [8]:
from sklearn import svm
from sklearn.metrics import classification_report

clf = svm.SVC(kernel="linear", random_state=42)
clf.fit(X_train, y_train)
y_pred_svm = clf.predict(X_test)

print("=== Hasil SVM ===")
print(classification_report(y_test, y_pred_svm))

=== Hasil SVM ===
               precision    recall  f1-score   support

      ekonomi       0.44      0.32      0.37        22
      hiburan       0.50      0.18      0.27        22
        hukum       0.21      0.38      0.27        13
internasional       0.67      0.17      0.28        23
   lingkungan       0.42      0.64      0.51        22
     olahraga       0.40      0.33      0.36        18
     otomotif       0.67      0.88      0.76        16
      politik       0.26      0.32      0.29        25
    sepakbola       0.50      0.68      0.58        19

     accuracy                           0.42       180
    macro avg       0.45      0.43      0.41       180
 weighted avg       0.45      0.42      0.40       180



## Siapkan data klasifikasi

In [9]:
from sklearn.model_selection import train_test_split

# Gunakan dataframe yang benar
X = X_topics
y = df_tempo_processed_csv["kategori_berita"]

# Split data menjadi data latih dan data uji
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


## Lakukan Klasifikasi

In [10]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

clf = LogisticRegression(max_iter=1000)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

print(classification_report(y_test, y_pred))


               precision    recall  f1-score   support

      ekonomi       0.41      0.32      0.36        22
      hiburan       0.50      0.27      0.35        22
        hukum       0.22      0.38      0.28        13
internasional       0.75      0.26      0.39        23
   lingkungan       0.41      0.64      0.50        22
     olahraga       0.33      0.28      0.30        18
     otomotif       0.70      0.88      0.78        16
      politik       0.28      0.32      0.30        25
    sepakbola       0.64      0.74      0.68        19

     accuracy                           0.44       180
    macro avg       0.47      0.45      0.44       180
 weighted avg       0.47      0.44      0.43       180



In [11]:
dominant_topics = []

for doc_topics in X_topics:
    # Get the index of the dominant topic for the current document
    dominant_topic_index = doc_topics.argmax()
    # Get the probability of the dominant topic
    dominant_topic_probability = doc_topics[dominant_topic_index]
    dominant_topics.append((dominant_topic_index + 1, round(dominant_topic_probability, 3)))

df_dominant = pd.DataFrame(dominant_topics, columns=["Topik Dominan", "Proporsi"])

# Concatenate with the original dataframe and the topic distribution dataframe
df_result = pd.concat([df_tempo_processed_csv.reset_index(drop=True), df_dominant.reset_index(drop=True), df_topics.reset_index(drop=True)], axis=1)

display(df_result[["judul_berita", "kategori_berita", "Topik Dominan", "Proporsi", "clean_text"]])

,judul_berita,kategori_berita,Topik Dominan,Proporsi,clean_text
0,Ketika Para Jenderal Ikut Defile di HUT ke-80 TNI,politik,9,0.982,ketika para jenderal ikut defile di hut ke tni...
1,Prabowo Minta Semua Pesantren Didata setelah P...,politik,8,0.795,prabowo minta semua pesantren didata setelah p...
2,Prabowo: Utamakan Kompetensi Prajurit Dibandin...,politik,9,0.828,prabowo utamakan kompetensi prajurit dibanding...
3,Kemenkomdigi: Permintaan Data ke TikTok Hanya ...,politik,1,0.684,kemenkomdigi permintaan data ke tiktok hanya u...
4,Megawati dan Jokowi Tak Hadir di HUT ke-80 TNI...,politik,9,0.995,megawati dan jokowi tak hadir di hut ke tni di...
...,...,...,...,...,...
895,Ketika Harry Kane Memecahkan Rekor Gol Cristia...,sepakbola,2,0.662,ketika harry kane memecahkan rekor gol cristia...
896,Peluang Timnas Indonesia Lewati Hadangan Arab ...,sepakbola,10,0.623,peluang timnas indonesia lewati hadangan arab ...
897,Seperti Apa Ketajaman Cristiano Ronaldo Bersam...,sepakbola,10,0.725,seperti apa ketajaman cristiano ronaldo bersam...
898,FIFA Jatuhkan Sanksi untuk Malaysia dan 7 Pema...,sepakbola,3,0.916,fifa jatuhkan sanksi untuk malaysia dan pemain...
